# Retail SKU visual retrieval → TFLite

This notebook is intentionally an embedding retriever. The catalog has one clean reference image for most SKUs, so a 3,000-class classifier would mostly memorize packshots. The exported model maps a query crop and catalog gallery images into the same vector space.

Upload the `catalog_manifest*.jsonl` shards as one private Kaggle Dataset before running. Keep the dataset private if product or store data is not public. Do not put API keys in the notebook.

In [ ]:
!pip -q install pillow pandas requests
import json, os, pathlib, hashlib, time
import numpy as np
import pandas as pd
import requests
from PIL import Image
import tensorflow as tf
print(tf.__version__)

In [ ]:
# Read all manifest shards from the private Kaggle Dataset.
INPUT_DIR = pathlib.Path('/kaggle/input/retail-ai-catalog')
MANIFESTS = sorted(INPUT_DIR.glob('catalog_manifest*.jsonl'))
if not MANIFESTS:
    raise FileNotFoundError('No catalog_manifest*.jsonl files found in /kaggle/input/retail-ai-catalog')
WORK = pathlib.Path('/kaggle/working/sku_retrieval')
IMAGE_DIR = WORK / 'catalog_images'
WORK.mkdir(parents=True, exist_ok=True)
IMAGE_DIR.mkdir(parents=True, exist_ok=True)
manifest = pd.concat([pd.read_json(path, lines=True) for path in MANIFESTS], ignore_index=True)
manifest = manifest.drop_duplicates('product_id').reset_index(drop=True)
print('catalog rows', len(manifest), 'unique image URLs', manifest.image_url.nunique())
manifest.head(2)

In [ ]:
# Download and validate reference images. A bad URL is excluded and logged.
session = requests.Session()
valid = []
failures = []
for row in manifest.to_dict('records'):
    target = IMAGE_DIR / (hashlib.sha1(row['product_id'].encode()).hexdigest() + '.jpg')
    try:
        if not target.exists():
            response = session.get(row['image_url'], timeout=20)
            response.raise_for_status()
            target.write_bytes(response.content)
        with Image.open(target) as image:
            image.convert('RGB').verify()
        row['local_path'] = str(target)
        valid.append(row)
    except Exception as exc:
        failures.append({'product_id': row['product_id'], 'error': type(exc).__name__})
print('valid', len(valid), 'failed', len(failures))
json.dump(failures, open(WORK / 'download_failures.json', 'w'), ensure_ascii=False, indent=2)
catalog = pd.DataFrame(valid)
catalog.to_json(WORK / 'valid_manifest.jsonl', orient='records', lines=True, force_ascii=False)

In [ ]:
# MobileNetV3Small is a compact backbone. The embedding head is the deployable output.
IMG_SIZE = (224, 224)
BATCH = 64
EMBEDDING_DIM = 256
num_classes = len(catalog)
class_to_id = {pid: i for i, pid in enumerate(catalog.product_id)}
paths = catalog.local_path.to_numpy()
labels = np.array([class_to_id[pid] for pid in catalog.product_id], dtype=np.int32)

augment = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.08),
    tf.keras.layers.RandomZoom(0.18),
    tf.keras.layers.RandomContrast(0.18),
], name='catalog_augmentation')

def load_image(path, label):
    image = tf.io.read_file(path)
    image = tf.image.decode_image(image, channels=3, expand_animations=False)
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32)
    return image, label

base_ds = tf.data.Dataset.from_tensor_slices((paths, labels)).map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
train_ds = base_ds.shuffle(min(len(paths), 4096), seed=42, reshuffle_each_iteration=True).batch(BATCH).prefetch(tf.data.AUTOTUNE)

inputs = tf.keras.Input(shape=(*IMG_SIZE, 3), name='image')
x = tf.keras.applications.mobilenet_v3.preprocess_input(inputs)
backbone = tf.keras.applications.MobileNetV3Small(include_top=False, weights='imagenet', pooling='avg')
backbone.trainable = False
x = backbone(x, training=False)
x = tf.keras.layers.Dense(EMBEDDING_DIM, activation=None, name='embedding_dense')(x)
embeddings = tf.keras.layers.Lambda(lambda value: tf.math.l2_normalize(value, axis=1), name='embedding')(x)
logits = tf.keras.layers.Dense(num_classes, name='sku_logits')(embeddings)
classifier = tf.keras.Model(inputs, logits)
embedder = tf.keras.Model(inputs, embeddings)
classifier.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True), metrics=['sparse_categorical_accuracy'])
classifier.summary()

In [ ]:
# This stage teaches the head with augmented views of the one-shot references.
# It is a bootstrap model, not a substitute for real phone photos.
history = classifier.fit(train_ds.map(lambda x, y: (augment(x, training=True), y)), epochs=8)

# Optional short backbone fine-tuning. Keep it small for Kaggle CPU/GPU portability.
backbone.trainable = True
for layer in backbone.layers[:-25]:
    layer.trainable = False
classifier.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True), metrics=['sparse_categorical_accuracy'])
classifier.fit(train_ds.map(lambda x, y: (augment(x, training=True), y)), epochs=3)

In [ ]:
# Synthetic retrieval check. Report this separately from real-phone accuracy.
gallery_images = base_ds.batch(BATCH).map(lambda x, y: x).prefetch(tf.data.AUTOTUNE)
gallery_vectors = embedder.predict(gallery_images, verbose=1)
gallery_vectors = gallery_vectors / np.maximum(np.linalg.norm(gallery_vectors, axis=1, keepdims=True), 1e-8)
query_ds = base_ds.batch(BATCH).map(lambda x, y: (augment(x, training=True), y)).prefetch(tf.data.AUTOTUNE)
query_vectors = []
query_labels = []
for images, y in query_ds:
    query_vectors.append(embedder(images, training=False).numpy())
    query_labels.append(y.numpy())
query_vectors = np.concatenate(query_vectors)
query_labels = np.concatenate(query_labels)
scores = query_vectors @ gallery_vectors.T
ranking = np.argsort(-scores, axis=1)
for k in (1, 5, 10):
    recall = np.mean([label in ranking[i, :k] for i, label in enumerate(query_labels)])
    print(f'synthetic Recall@{k}: {recall:.4f}')

In [ ]:
# Export the small TFLite embedder and its gallery.
export_dir = WORK / 'export'
export_dir.mkdir(exist_ok=True)
converter = tf.lite.TFLiteConverter.from_keras_model(embedder)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()
(export_dir / 'sku_embedder.tflite').write_bytes(tflite_model)
np.save(export_dir / 'gallery_embeddings.npy', gallery_vectors.astype('float32'))
catalog[['product_id','name','category','brand','shelf_code','shelf_friendly','image_url']].to_json(export_dir / 'gallery_metadata.json', orient='records', force_ascii=False, indent=2)
json.dump({'input_size': list(IMG_SIZE), 'embedding_dim': EMBEDDING_DIM, 'num_gallery_rows': len(catalog), 'metric': 'cosine', 'validation': 'synthetic augmentation only'}, open(export_dir / 'model_card.json', 'w'), ensure_ascii=False, indent=2)
print('TFLite bytes', len(tflite_model), 'gallery rows', len(catalog))

## Required next data collection

Before enabling automatic shelf directions, collect product-disjoint phone photos and measure Recall@1, Recall@5, false-match rate, and abstention rate. A full shelf photo needs a detector/cropper before this embedder. If the top-1/top-2 margin is small, show both candidates and ask for confirmation.